
# Reusable Project Template: Discrete Response Regression

Copy this notebook as the starting point for a new project involving a **binary**, **multi-class**,
or **count** outcome. Fill in the `# >>> EDIT` sections and delete whichever model-family sections
you don't need. Helper functions are written generically so you can import them into other notebooks
too (see the bottom of the notebook for a "save as .py module" snippet).



---
## 0. Setup


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families.family import NegativeBinomial

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_absolute_percentage_error, mean_squared_error,
)

pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)



---
## 1. Load and inspect data

> **>>> EDIT:** replace this cell with your actual data loading (`pd.read_csv`, a database query, an
> API pull, etc).


In [ ]:

# >>> EDIT: load your dataset
df = pd.read_csv('your_data.csv')   # <-- change me

df.head()


In [ ]:

# Standard first-look checklist -- keep this every time
print(df.shape)
df.info()
df.describe(include='all').T
df.isna().sum()



---
## 2. Define the problem type

> **>>> EDIT:** set `TARGET`, `FEATURES`, and `PROBLEM_TYPE`. `PROBLEM_TYPE` controls which section
> below you should run: `'binary'`, `'multinomial'`, `'poisson'`, or `'negative_binomial'`.
> If you're not sure whether it's Poisson or negative binomial, run the dispersion check in Section 3
> first, then come back and set this.


In [ ]:

TARGET = 'y'                       # >>> EDIT
FEATURES = ['x1', 'x2', 'x3']      # >>> EDIT
PROBLEM_TYPE = 'binary'            # >>> EDIT: 'binary' | 'multinomial' | 'poisson' | 'negative_binomial'

assert PROBLEM_TYPE in {'binary', 'multinomial', 'poisson', 'negative_binomial'}



---
## 3. Diagnostics before modeling

Run whichever of these applies.


In [ ]:

# For binary / multinomial targets: check class balance
if PROBLEM_TYPE in ('binary', 'multinomial'):
    print(df[TARGET].value_counts(normalize=True))
    df[TARGET].value_counts().plot(kind='bar', title='Class balance')
    plt.show()


In [ ]:

# For count targets: check equidispersion (mean vs variance) BEFORE choosing Poisson vs NegBin
if PROBLEM_TYPE in ('poisson', 'negative_binomial'):
    mean_y = df[TARGET].mean()
    var_y = df[TARGET].var()
    print(f'mean={mean_y:.3f}  variance={var_y:.3f}  ratio(var/mean)={var_y/mean_y:.2f}')
    plt.hist(df[TARGET], bins=30)
    plt.title('Target distribution')
    plt.show()
    print('Rule of thumb: ratio >> 1 suggests overdispersion -> use negative_binomial instead of poisson.')



---
## 4. Train / test split


In [ ]:

stratify_col = df[TARGET] if PROBLEM_TYPE in ('binary', 'multinomial') else None
train_df, test_df = train_test_split(df, test_size=0.25, random_state=RANDOM_STATE, stratify=stratify_col)

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]



---
## 5A. Binary outcome (logit / probit) — run this section if `PROBLEM_TYPE == 'binary'`


In [ ]:

if PROBLEM_TYPE == 'binary':
    formula = f"{TARGET} ~ " + " + ".join(FEATURES)   # >>> EDIT if you need interactions/transforms

    logit_model = smf.logit(formula, data=train_df).fit()
    print(logit_model.summary())

    print('\nOdds ratios:')
    print(np.exp(logit_model.params))

    y_prob = logit_model.predict(test_df[FEATURES])
    y_pred = (y_prob >= 0.5).astype(int)             # >>> EDIT threshold if needed

    print('\nAccuracy :', accuracy_score(y_test, y_pred))
    print('Precision:', precision_score(y_test, y_pred))
    print('Recall   :', recall_score(y_test, y_pred))
    print('F1       :', f1_score(y_test, y_pred))
    print('AUC      :', roc_auc_score(y_test, y_prob))
    print('McFadden pseudo-R2:', logit_model.prsquared)

    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot()
    plt.show()

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label='logit'); plt.plot([0, 1], [0, 1], '--', color='gray')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.legend(); plt.show()



---
## 5B. Multi-class outcome (multinomial logit) — run this section if `PROBLEM_TYPE == 'multinomial'`


In [ ]:

if PROBLEM_TYPE == 'multinomial':
    model_sk = LogisticRegression(solver='lbfgs', max_iter=1000)  # multinomial is automatic for multi-class y in modern sklearn
    model_sk.fit(X_train, y_train)
    pred_sk = model_sk.predict(X_test)
    print('sklearn test accuracy:', accuracy_score(y_test, pred_sk))

    X_train_c = sm.add_constant(X_train)
    X_test_c = sm.add_constant(X_test)
    model_stat = sm.MNLogit(y_train, X_train_c).fit(method='bfgs')
    print(model_stat.summary())

    pred_stat = np.asarray(model_stat.predict(X_test_c)).argmax(1)
    print('statsmodels test accuracy:', accuracy_score(y_test, pred_stat))

    print('\nRelative risk ratios (statsmodels, vs baseline category):')
    print(np.exp(model_stat.params))

    ConfusionMatrixDisplay(confusion_matrix(y_test, pred_sk)).plot()
    plt.title('sklearn model'); plt.show()



---
## 5C. Count outcome — Poisson — run this section if `PROBLEM_TYPE == 'poisson'`


In [ ]:

if PROBLEM_TYPE == 'poisson':
    X_train_c = sm.add_constant(X_train)
    X_test_c = sm.add_constant(X_test)

    poisson_model = sm.Poisson(y_train, X_train_c).fit()
    print(poisson_model.summary())

    print('\nMultiplicative effects (exp of coefficients):')
    print(np.exp(poisson_model.params))

    preds_train = poisson_model.predict(X_train_c)
    preds_test = poisson_model.predict(X_test_c)

    print('\nTrain MAE :', mean_absolute_error(y_train, preds_train))
    print('Test  MAE :', mean_absolute_error(y_test, preds_test))
    print('Train MAPE:', mean_absolute_percentage_error(y_train, preds_train))
    print('Test  MAPE:', mean_absolute_percentage_error(y_test, preds_test))

    print('\nDispersion check (deviance/df_resid, >>1 suggests overdispersion):',
          poisson_model.deviance / poisson_model.df_resid if hasattr(poisson_model, 'deviance') else 'n/a for discrete Poisson - refit with sm.GLM to get deviance')



---
## 5D. Count outcome — Negative Binomial — run this section if `PROBLEM_TYPE == 'negative_binomial'`


In [ ]:

if PROBLEM_TYPE == 'negative_binomial':
    X_train_c = sm.add_constant(X_train)
    X_test_c = sm.add_constant(X_test)

    # Step 1: Poisson fit for mu_hat
    poisson_model = sm.GLM(y_train, X_train_c, family=sm.families.Poisson()).fit()

    # Step 2: auxiliary OLS regression to estimate alpha
    df_aux = pd.DataFrame({'mu_hat': poisson_model.mu, 'y': y_train.values})
    df_aux['y_auxiliary'] = ((df_aux['y'] - df_aux['mu_hat'])**2 - df_aux['mu_hat']) / df_aux['mu_hat']
    ols_model = smf.ols('y_auxiliary ~ mu_hat - 1', df_aux).fit()
    alpha_hat = ols_model.params.iloc[0]
    print('Estimated alpha:', alpha_hat)
    print(ols_model.summary())

    # Step 3: negative binomial fit
    nb_model = sm.GLM(y_train, X_train_c, family=NegativeBinomial(alpha=alpha_hat)).fit()
    print(nb_model.summary())

    preds_train = nb_model.predict(X_train_c)
    preds_test = nb_model.predict(X_test_c)

    print('\nTrain RMSE:', np.sqrt(mean_squared_error(y_train, preds_train)))
    print('Test  RMSE:', np.sqrt(mean_squared_error(y_test, preds_test)))

    print('\nStandard error comparison (Poisson vs NegBin) -- NegBin should generally be wider if overdispersion is real:')
    print(pd.DataFrame({'poisson_se': poisson_model.bse, 'negbin_se': nb_model.bse}))



---
## 6. Write-up

> **>>> EDIT:** replace with your own findings.

- **Model chosen:**
- **Key coefficients and their interpretation:**
- **Performance on held-out data:**
- **Known limitations / caveats:** (see `06_Background_Theory.md` for a checklist)
- **Recommendation / next steps:**



---
## 7. (Optional) Turn the diagnostics/eval code into reusable functions

If you find yourself copy-pasting the evaluation blocks above across projects, promote them into a
small local module, e.g. `model_utils.py`:

```python
def evaluate_binary_model(model, X_test, y_test, threshold=0.5):
    y_prob = model.predict(X_test)
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_prob),
    }

def estimate_negative_binomial_alpha(y_train, X_train_const):
    poisson_model = sm.GLM(y_train, X_train_const, family=sm.families.Poisson()).fit()
    df_aux = pd.DataFrame({'mu_hat': poisson_model.mu, 'y': y_train.values})
    df_aux['y_auxiliary'] = ((df_aux['y'] - df_aux['mu_hat'])**2 - df_aux['mu_hat']) / df_aux['mu_hat']
    ols_model = smf.ols('y_auxiliary ~ mu_hat - 1', df_aux).fit()
    return ols_model.params.iloc[0], ols_model
```

Then `from model_utils import evaluate_binary_model, estimate_negative_binomial_alpha` at the top of
future notebooks.
